# SG TimeBank Lab: TIME Credits on Chain

## What We Will Do

1. **Test** the `SGTimeCredit` smart contract with Forge (30 tests, including a fuzz test).
2. **Deploy** it to a local Anvil blockchain, using the same Scaffold-ETH deploy pattern as Lab 2.
3. **Run the full volunteer flow** with `cast`: tap in, tap out, sign-offs, the 14-day dispute window, minting, redemption, gift and donation.
4. **(Optional) Deploy to BSC Testnet** and view the contract on BscScan, as in Lab 4.

All organisations, partners and people in this lab are hypothetical.

### Pre-Class Setup

Same tools and versions as Labs 1, 2 and 4:

| Tool | Recommended version |
| --- | --- |
| Visual Studio Code | Latest stable |
| Python | 3.12.10 |
| Node.js | 24.19.0 (not needed for this lab, but used by Scaffold-ETH) |
| Git | 2.53.0 |
| Foundry (Forge, Anvil, Cast) | 1.8.1 |

Folder layout expected by this notebook:

```
SG_TimeBank_Lab.ipynb
sgtimebank/
├── contracts/SGTimeCredit.sol      ← the TIME credit contract
├── test/SGTimeCredit.t.sol         ← 30 Forge tests
├── script/Deploy.s.sol             ← deploys (Scaffold-ETH pattern, as in Lab 2)
├── script/DeploySGTimeCredit.s.sol
├── script/DeployHelpers.s.sol      ← unchanged from Scaffold-ETH 2
├── script/SeedDemo.s.sol           ← registers demo organisations, tags, partners, pool
└── foundry.toml
```

To use it inside your Scaffold-ETH project from Lab 2 instead, copy `contracts/`, `test/` and the three `script/` files into `packages/foundry/` and add `DeploySGTimeCredit` to that project's `Deploy.s.sol`.

---
## Step 1 — Initialization

- Prepare the environment on Windows, macOS or Linux
- Prepare helper functions to start Anvil and run Forge / Cast

In [1]:
import json, locale, os, shutil, socket, subprocess, sys, time
from pathlib import Path
from urllib.request import urlopen, Request
from urllib.error import URLError

IS_WINDOWS = sys.platform == 'win32'
IS_MACOS = sys.platform == 'darwin'
os.environ['NO_COLOR'] = '1'

foundry_bin = Path.home() / '.foundry' / 'bin'
if foundry_bin.is_dir() and str(foundry_bin) not in os.environ.get('PATH', '').split(os.pathsep):
    os.environ['PATH'] = f"{foundry_bin}{os.pathsep}{os.environ.get('PATH', '')}"

PROJECT_NAME = 'sgtimebank'
start_dir = Path.cwd().resolve()
PROJECT_DIR = start_dir if start_dir.name == PROJECT_NAME else start_dir / PROJECT_NAME
if not (PROJECT_DIR / 'foundry.toml').exists():
    raise FileNotFoundError(f'Cannot find {PROJECT_DIR / "foundry.toml"}. Put this notebook next to the sgtimebank folder.')

def _decode(data):
    if not data:
        return ''
    for enc in ('utf-8', locale.getpreferredencoding(False), 'gb18030'):
        try:
            return data.decode(enc)
        except (UnicodeDecodeError, LookupError):
            pass
    return data.decode('utf-8', errors='replace')

def run(cmd, cwd=None, quiet=False):
    """Run a command (list or string). Returns (returncode, stdout, stderr)."""
    result = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, cwd=cwd or PROJECT_DIR)
    out, err = _decode(result.stdout).strip(), _decode(result.stderr).strip()
    if not quiet:
        if out: print(out)
        if err and err != out: print(err)
    return result.returncode, out, err

_launched = []

def open_terminal(title, command, cwd):
    """Open a new terminal window (as in Lab 2). Falls back to a background process."""
    cwd_str = str(cwd)
    if IS_WINDOWS:
        if shutil.which('wt'):
            proc = subprocess.Popen(['wt.exe', '--title', title, 'cmd', '/k', f'cd /d "{cwd_str}" && {command}'])
        else:
            proc = subprocess.Popen(f'start "{title}" cmd /k "cd /d \\"{cwd_str}\\" && {command}"', shell=True)
    elif IS_MACOS:
        script = f'tell application "Terminal"\n    do script "cd \\"{cwd_str}\\" && {command}"\n    activate\nend tell'
        proc = subprocess.Popen(['osascript', '-e', script])
    else:
        term = next((t for t in ('gnome-terminal', 'xterm', 'konsole') if shutil.which(t)), None)
        if term == 'gnome-terminal':
            proc = subprocess.Popen([term, '--title', title, '--', 'bash', '-c', f'cd "{cwd_str}" && {command}; exec bash'])
        elif term:
            proc = subprocess.Popen([term, '-title', title, '-e', 'bash', '-c', f'cd "{cwd_str}" && {command}; exec bash'])
        else:
            proc = subprocess.Popen(command.split(), cwd=cwd_str, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    _launched.append(proc)
    return proc

def check_port(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        try:
            s.connect(('127.0.0.1', port)); return True
        except OSError:
            return False

def wait_for_rpc(url, timeout=30):
    payload = b'{"jsonrpc":"2.0","method":"eth_blockNumber","params":[],"id":1}'
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urlopen(Request(url, data=payload, headers={'Content-Type': 'application/json'}), timeout=2) as r:
                if r.status == 200: return True
        except (URLError, OSError):
            pass
        time.sleep(1)
    return False

platform_name = 'Windows' if IS_WINDOWS else ('macOS' if IS_MACOS else 'Linux')
print(f'Initialized Successfully!!! ({platform_name}). Project directory: {PROJECT_DIR}')

Initialized Successfully!!! (Linux test run). Project directory: ./sgtimebank


---
## Step 2 — Environment Check

In [2]:
all_ok = True
for name, cmd in [('Python', f'"{sys.executable}" --version'), ('Git', 'git --version'),
                  ('Anvil', 'anvil --version'), ('Forge', 'forge --version'), ('Cast', 'cast --version')]:
    rc, out, err = run(cmd, quiet=True)
    version = (out or err or 'not found').splitlines()[0]
    all_ok &= rc == 0
    print(f"  {'[OK]' if rc == 0 else '[MISSING]':10s} {name:7s} {version}")
print('\nAll tools are available.' if all_ok else '\nSome tools are missing. Install Foundry 1.8.1 first.')

  [OK]       Python  Python 3.13.18
  [OK]       Git     git version 2.43.0
  [OK]       Anvil   anvil Version: 1.8.1
  [OK]       Forge   forge Version: 1.8.1
  [OK]       Cast    cast Version: 1.8.1

All tools are available.


---
## Step 3 — Install forge-std

Forge tests and scripts need the `forge-std` library (same command as Lab 2).

In [3]:
forge_std = PROJECT_DIR / 'lib' / 'forge-std'
if forge_std.exists():
    print('forge-std already installed.')
else:
    print('Installing forge-std...')
    rc, _, _ = run('forge install foundry-rs/forge-std --no-git')
    if rc != 0:
        raise RuntimeError('Failed to install forge-std. Check your internet connection.')
print('Ready.')

Installing forge-std...


Installing forge-std in ./sgtimebank/lib/forge-std (url: https://github.com/foundry-rs/forge-std, tag: None)
Cloning into './sgtimebank/lib/forge-std'...
    Installed forge-std
Ready.


---
## Step 4 — Understanding the SGTimeCredit Contract

`contracts/SGTimeCredit.sol` uses the same building blocks as the CrowdFund contract from Lab 2:

| Lab 2 CrowdFund | SGTimeCredit | Purpose |
|---|---|---|
| `mapping(address => uint256) balances` | `mapping(address => uint256) balanceOf` | Credits held by each volunteer |
| `event Contribution` | `event Minted`, `Redeemed`, `Gifted`, `Donated` ... | Public, auditable history |
| `deadline = block.timestamp + ...` | `readyAt + 14 days` dispute window | Time-based rule enforced by code |
| `execute()` after the deadline | `finalise()` after the dispute window | Anyone can trigger; no trusted organiser |
| `error TooEarly(...)` | `error DisputeWindowOpen(...)` | Custom errors explain why a call failed |

**Rules enforced on chain**

- **Minting:** NFC tap-in and tap-out at a registered venue tag, then the **coordinator** plus a **beneficiary or peer** sign. Credits mint after a **14-day dispute window**.
- **Skill tiers:** General 1.0x, Skilled 1.5x (skill certificate), Professional 2.0x (licence).
- **Caps:** 8 credits per work day and 60 per calendar month (Singapore time).
- **Earn-and-burn:** partners accept credits; volunteers with 20+ verified hours at a partner's organisation pay 0.8 instead of 1.0 per service hour.
- **No trading:** credits move only as a gift to a registered family member or a donation to a community pool. There is no `transfer()`.

Amounts use 2 decimals: `600` means 6.00 credits.

---
## Step 5 — Run the Tests

In [4]:
rc, out, err = run('forge test', quiet=True)
lines = (out + '\n' + err).splitlines()
for line in lines:
    if line.strip().startswith(('[PASS]', '[FAIL')):
        print(' ', line.strip())
for line in lines:
    if 'Suite result' in line:
        print('\n' + line.strip())

  [PASS] testFuzz_QuarterHourRounding(uint256) (runs: 256, μ: 228895, ~: 229103)
  [PASS] test_CannotSignOwnHours() (gas: 242349)
  [PASS] test_CapsApplyPerWorkDayNotMintDay() (gas: 986871)
  [PASS] test_CoordinatorAloneIsNotEnough() (gas: 381982)
  [PASS] test_CoordinatorOfAnotherOrgCannotSign() (gas: 254396)
  [PASS] test_CreditsCannotBeTransferredFreely() (gas: 7933)
  [PASS] test_DailyCapLimitsToEightCredits() (gas: 932714)
  [PASS] test_DisputedSessionNeverMints() (gas: 417275)
  [PASS] test_DonationsGoToPool() (gas: 607281)
  [PASS] test_FamilyCannotSign() (gas: 246700)
  [PASS] test_GeneralTierMintsOneCreditPerHour() (gas: 541220)
  [PASS] test_GiftsOnlyToFamily() (gas: 636614)
  [PASS] test_LoyaltyAfterTwentyHoursCosts1point6For2Hours() (gas: 1546192)
  [PASS] test_LoyaltyBelowThresholdIsStandardRate() (gas: 1009430)
  [PASS] test_MonthKeyUsesSingaporeTime() (gas: 23458)
  [PASS] test_MonthlyCapOfSixty() (gas: 3520622)
  [PASS] test_NothingMintsDuringDisputeWindow() (gas: 39344

---
## Step 6 — Start Anvil Local Blockchain

In [5]:
ANVIL_URL = 'http://127.0.0.1:8545'
if check_port(8545) and wait_for_rpc(ANVIL_URL, timeout=3):
    print('Anvil is already running.')
else:
    print('Starting Anvil...')
    open_terminal('Anvil', 'anvil', PROJECT_DIR)
    print(f'Anvil is ready at {ANVIL_URL} (chain ID 31337)' if wait_for_rpc(ANVIL_URL) else 'ERROR: Anvil did not start.')

Starting Anvil...


Anvil is ready at http://127.0.0.1:8545 (chain ID 31337)


---
## Step 7 — Deploy and Seed on Anvil

1. `Deploy.s.sol` deploys `SGTimeCredit` (deploy-only, like `DeployCrowdFund.s.sol`).
2. `SeedDemo.s.sol` registers two hypothetical organisations, their venue tags, two partners and a community pool.

> This uses Anvil's publicly known account 0 key. Never use it on a real network.

In [6]:
ANVIL_PK = '0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80'
(PROJECT_DIR / 'deployments').mkdir(exist_ok=True)  # Scaffold-ETH's DeployHelpers writes here

rc, out, err = run(['forge', 'script', 'script/Deploy.s.sol', '--rpc-url', 'localhost',
                    '--broadcast', '--private-key', ANVIL_PK], quiet=True)
if rc != 0:
    print((err or out)[-2000:]); raise RuntimeError('Deploy failed. Is Anvil running? (Step 6)')
deployments = json.loads((PROJECT_DIR / 'deployments' / '31337.json').read_text())
SGT_ADDRESS = next(addr for addr, name in deployments.items() if name == 'SGTimeCredit')
print(f'SGTimeCredit deployed at: {SGT_ADDRESS}')

env = dict(os.environ, SGT_ADDRESS=SGT_ADDRESS)
r = subprocess.run(['forge', 'script', 'script/SeedDemo.s.sol', '--rpc-url', 'localhost',
                    '--broadcast', '--private-key', ANVIL_PK], cwd=PROJECT_DIR, capture_output=True, env=env)
seeded = [l.strip() for l in _decode(r.stdout).splitlines() if 'Seeded' in l]
print(seeded[0] if seeded else _decode(r.stderr)[-1500:])

SGTimeCredit deployed at: 0x5FbDB2315678afecb367f032d93F642f64180aa3


Seeded: orgs 2 partners 2


---
## Step 8 — Run the Volunteer Flow with Cast

We use Anvil's test accounts as the people in the story:

| Account | Role |
|---|---|
| 0 | Operator (admin) |
| 1 | Aisha, general volunteer |
| 2 | Ravi, licensed physiotherapist |
| 3 | Mei, peer volunteer (skill certificate) |
| 4 | Kumar, Aisha's family member |
| 5 | Coordinator, Community Centre North (org 1) |
| 6 | Coordinator, Clinic X (org 2) |

First, the helper functions:

In [7]:
MNEMONIC = 'test test test test test test test test test test test junk'
_keys = {}
def key(i):
    if i not in _keys:
        _, out, _ = run(['cast', 'wallet', 'private-key', '--mnemonic', MNEMONIC, '--mnemonic-index', str(i)], quiet=True)
        _keys[i] = out.strip()
    return _keys[i]

def addr(i):
    _, out, _ = run(['cast', 'wallet', 'address', '--private-key', key(i)], quiet=True)
    return out.strip()

def send(i, sig, *args, expect_error=None):
    """Send a transaction from account i. If expect_error is set, report whether it was refused."""
    rc, out, err = run(['cast', 'send', SGT_ADDRESS, sig, *map(str, args),
                        '--rpc-url', ANVIL_URL, '--private-key', key(i)], quiet=True)
    if expect_error:
        found = expect_error in (out + err)
        print(f'  refused with {expect_error}' if rc != 0 and found else f'  UNEXPECTED result (rc={rc})')
    elif rc != 0:
        raise RuntimeError(f'{sig} failed:\n{err[-800:]}')

def call(sig, *args):
    _, out, _ = run(['cast', 'call', SGT_ADDRESS, sig, *map(str, args), '--rpc-url', ANVIL_URL], quiet=True)
    return out.strip().split()[0]

def credits(i):
    return int(call('balanceOf(address)(uint256)', addr(i))) / 100

def warp(seconds):
    run(['cast', 'rpc', 'evm_increaseTime', str(seconds), '--rpc-url', ANVIL_URL], quiet=True)
    run(['cast', 'rpc', 'evm_mine', '--rpc-url', ANVIL_URL], quiet=True)

STATUS = ['None', 'Open', 'AwaitingSignoff', 'Ready', 'Disputed', 'Minted']
def status(session_id):
    _, out, _ = run(['cast', 'call', SGT_ADDRESS,
                     'sessions(uint256)(address,uint256,uint8,uint64,uint64,uint64,uint32,uint8,uint256,bool,uint8)',
                     str(session_id), '--rpc-url', ANVIL_URL], quiet=True)
    return STATUS[int(out.strip().splitlines()[-1])]

def keccak(text):
    _, out, _ = run(['cast', 'keccak', text], quiet=True)
    return out.strip()

TAG_CC, TAG_CLINIC = keccak('tag-cc-1'), keccak('tag-clinic')
print('Helpers ready.')

Helpers ready.


### 8.1 Onboarding (operator)

The operator verifies each volunteer once. The contract stores a **hash** of their national ID so one person cannot register two wallets, without putting the ID itself on chain.

In [8]:
send(0, 'verifyVolunteer(address,bytes32,bool,bool)', addr(1), keccak('S1111111A'), 'false', 'false')
send(0, 'verifyVolunteer(address,bytes32,bool,bool)', addr(2), keccak('S2222222B'), 'false', 'true')
send(0, 'verifyVolunteer(address,bytes32,bool,bool)', addr(3), keccak('S3333333C'), 'true', 'false')
send(0, 'verifyVolunteer(address,bytes32,bool,bool)', addr(4), keccak('S4444444D'), 'false', 'false')
send(0, 'setCoordinator(uint256,address,bool)', 1, addr(5), 'true')
send(0, 'setCoordinator(uint256,address,bool)', 2, addr(6), 'true')
send(0, 'linkFamily(address,address)', addr(1), addr(4))
print('4 volunteers verified (Ravi licensed, Mei skill-certified), 2 coordinators, Aisha-Kumar family link.')

print('\nSomeone tries to register a second wallet with Aisha\'s ID:')
send(0, 'verifyVolunteer(address,bytes32,bool,bool)', addr(7), keccak('S1111111A'), 'false', 'false',
     expect_error='IdentityAlreadyUsed')

4 volunteers verified (Ravi licensed, Mei skill-certified), 2 coordinators, Aisha-Kumar family link.

Someone tries to register a second wallet with Aisha's ID:
  refused with IdentityAlreadyUsed


### 8.2 Ravi volunteers 3 hours of physiotherapy (Professional tier, 2.0x)

In [9]:
send(2, 'checkIn(uint256,bytes32,uint8)', 2, TAG_CLINIC, 2)
S1 = int(call('sessionCount()(uint256)'))
warp(3 * 3600)
send(2, 'checkOut(uint256,bytes32)', S1, TAG_CLINIC)
send(6, 'sign(uint256,uint8)', S1, 1)
print(f'Session {S1}: coordinator signed -> {status(S1)}')
send(3, 'sign(uint256,uint8)', S1, 3)
print(f'Session {S1}: peer signed        -> {status(S1)}')

print('\nAisha (no licence) tries the Professional tier:')
send(1, 'checkIn(uint256,bytes32,uint8)', 2, TAG_CLINIC, 2, expect_error='MissingCredential')

Session 1: coordinator signed -> AwaitingSignoff


Session 1: peer signed        -> Ready

Aisha (no licence) tries the Professional tier:
  refused with MissingCredential


### 8.3 Aisha volunteers 4 hours (General tier), and the safeguards

In [10]:
send(1, 'checkIn(uint256,bytes32,uint8)', 1, TAG_CC, 0)
S2 = int(call('sessionCount()(uint256)'))
warp(4 * 3600)
send(1, 'checkOut(uint256,bytes32)', S2, TAG_CC)
send(5, 'sign(uint256,uint8)', S2, 1)

print('Aisha tries to sign her own hours:')
send(1, 'sign(uint256,uint8)', S2, 3, expect_error='SelfSign')
print('Kumar (family) tries to sign Aisha\'s hours:')
send(4, 'sign(uint256,uint8)', S2, 3, expect_error='FamilyCannotSign')
send(3, 'sign(uint256,uint8)', S2, 3)
print(f'Mei signs as peer -> {status(S2)}')

print('\nTry to mint before the 14-day dispute window closes:')
send(0, 'finalise(uint256)', S1, expect_error='DisputeWindowOpen')

Aisha tries to sign her own hours:
  refused with SelfSign
Kumar (family) tries to sign Aisha's hours:
  refused with FamilyCannotSign


Mei signs as peer -> Ready

Try to mint before the 14-day dispute window closes:
  refused with DisputeWindowOpen


### 8.4 Fast-forward 14 days and mint

Like `execute()` in CrowdFund, **anyone** can call `finalise()` once the window has passed.

In [11]:
warp(14 * 86400)
send(0, 'finalise(uint256)', S1)
send(0, 'finalise(uint256)', S2)
print(f'Ravi:  {credits(2):.2f} credits  (3 h x 2.0)')
print(f'Aisha: {credits(1):.2f} credits  (4 h x 1.0)')

Ravi:  6.00 credits  (3 h x 2.0)
Aisha: 4.00 credits  (4 h x 1.0)


### 8.5 Earn-and-burn: redeem, gift, donate

In [12]:
rate = call('redeemRate(address,uint256)(uint256,bool)', addr(2), 1)
print(f'Ravi\'s rate at Clinic X Physio: {int(rate)/100:.2f} credit per hour (loyalty starts at 20 verified hours)')
send(2, 'redeem(uint256,uint256)', 1, 4)  # 4 quarter-hours = 1 hour
print(f'Ravi redeems 1 h of physio -> {credits(2):.2f} left')

send(1, 'gift(address,uint256)', addr(4), 150)
print(f'Aisha gifts 1.50 to Kumar  -> Kumar {credits(4):.2f}, Aisha {credits(1):.2f}')
send(1, 'donate(uint256,uint256)', 1, 100)
print(f'Aisha donates 1.00 to the Elder Care Pool -> pool {int(call("poolBalance(uint256)(uint256)", 1))/100:.2f}')

print('\nAisha tries to send credits to Mei (not family):')
send(1, 'gift(address,uint256)', addr(3), 50, expect_error='NotFamily')

minted, burned = int(call('totalMinted()(uint256)')), int(call('totalBurned()(uint256)'))
print(f'\nSupply: minted {minted/100:.2f}, burned {burned/100:.2f}, circulating {(minted-burned)/100:.2f}')

Ravi's rate at Clinic X Physio: 1.00 credit per hour (loyalty starts at 20 verified hours)


Ravi redeems 1 h of physio -> 5.00 left


Aisha gifts 1.50 to Kumar  -> Kumar 1.50, Aisha 2.50


Aisha donates 1.00 to the Elder Care Pool -> pool 1.00

Aisha tries to send credits to Mei (not family):
  refused with NotFamily

Supply: minted 10.00, burned 1.00, circulating 9.00


### 8.6 Read the event log

Events are the public record that the prototype's Ledger screen shows.

In [13]:
_, out, _ = run(['cast', 'logs', '--address', SGT_ADDRESS, 'Minted(uint256 indexed,address indexed,uint256,bool)',
                 '--from-block', '0', '--rpc-url', ANVIL_URL], quiet=True)
hashes = [l.split()[-1] for l in out.splitlines() if 'transactionHash' in l]
print(f'{len(hashes)} Minted events:')
for h in hashes:
    print(' ', h)

2 Minted events:
  0xc1e65f30e0d64877cce1454b7c8f068ca2bfb82954e31dd2c5e5f1ac099da2e8
  0x0571f092bb45380a9b3c8fd4d1d7a2261fda16f907754463ed888fee2ddeb738


---
## Step 9 — (Optional) Deploy to BSC Testnet

Same process as Lab 4 Step 10:

1. Create a **separate test wallet** in MetaMask and get tBNB from a faucet (see the faucet table in Lab 4).
2. Set the wallet's private key in an environment variable before starting Jupyter, for example `set SGT_TESTNET_PK=0x...` on Windows. Do not paste a real wallet's key into a notebook.
3. Change `RUN_TESTNET` to `True` and run the cell.

Deployment uses about 2.5 million gas. At 0.1 gwei that is about 0.00025 tBNB.

In [14]:
RUN_TESTNET = False
BSC_TESTNET_RPC_URL = 'https://bsc-testnet-dataseed.bnbchain.org'
BSCSCAN_BASE_URL = 'https://testnet.bscscan.com'

if not RUN_TESTNET:
    print('Skipped. Set RUN_TESTNET = True to deploy to BSC Testnet (chain ID 97).')
else:
    (PROJECT_DIR / 'deployments').mkdir(exist_ok=True)
    pk = os.environ.get('SGT_TESTNET_PK')
    if not pk:
        from getpass import getpass
        pk = getpass('Test wallet private key (input hidden): ').strip()
    rc, out, err = run(['forge', 'script', 'script/Deploy.s.sol', '--rpc-url', BSC_TESTNET_RPC_URL,
                        '--broadcast', '--private-key', pk], quiet=True)
    if rc != 0:
        print((err or out)[-2000:])
    else:
        d = json.loads((PROJECT_DIR / 'deployments' / '97.json').read_text())
        TESTNET_ADDRESS = next(a for a, n in d.items() if n == 'SGTimeCredit')
        print(f'SGTimeCredit on BSC Testnet: {TESTNET_ADDRESS}')
        print(f'{BSCSCAN_BASE_URL}/address/{TESTNET_ADDRESS}')
        r = subprocess.run(['forge', 'script', 'script/SeedDemo.s.sol', '--rpc-url', BSC_TESTNET_RPC_URL,
                            '--broadcast', '--slow', '--private-key', pk], cwd=PROJECT_DIR, capture_output=True,
                           env=dict(os.environ, SGT_ADDRESS=TESTNET_ADDRESS))
        print('Seeded demo data.' if r.returncode == 0 else _decode(r.stderr)[-1500:])

Skipped. Set RUN_TESTNET = True to deploy to BSC Testnet (chain ID 97).


---
## End: Cleanup

Stops the Anvil process started by this notebook.

In [15]:
for proc in _launched:
    try:
        proc.terminate()
    except OSError:
        pass
_launched.clear()
print('Cleanup complete. If Anvil was opened in its own terminal window, close that window too.')

Cleanup complete. If Anvil was opened in its own terminal window, close that window too.


---
## Summary

- **Tested:** 30 Forge tests cover every rule in the tokenomics design.
- **Deployed:** the same Scaffold-ETH deploy pattern as Lab 2, then demo data seeded.
- **Demonstrated on chain:** two-party sign-off, the 14-day dispute window, skill-tier minting, earn-and-burn redemption, family-only gifts and pool donations.

**Trust the Protocol, Not the Organizer.**